# Example: Tokenizing the Sarcasm Dataset
This example will familiarize students with working with unstructured text, particularly the generation of a vocabulary model and the tokenization of text, i.e., the conversion of sentences into a mathematical representation.

### Learning Objectives
First, we'll setup the computational environment by including the `Include.jl` file and loading any needed resources, e.g., the dataset that we'll explore. Then, we'll tokenize the text data, i.e., convert the text into a mathematical representation that can be used in machine learning models. 

Let's go!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.
* The [include command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/). 

In [41]:
include("Include.jl");

In addition to standard Julia libraries, we'll also use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types and data used in this material. 

### Data
Let's load a public dataset of headlines that have been curated as either __sarcastic__ or __not sarcastic__. The dataset we'll use is [publically available on Kaggle](https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection) and is also discussed in the publications:
1. Misra, Rishabh and Prahal Arora. "Sarcasm Detection using News Headlines Dataset." AI Open (2023).
2. Misra, Rishabh and Jigyasa Grover. "Sculpting Data for ML: The first act of Machine Learning." ISBN 9798585463570 (2021).

We've packaged the sarcasm dataset in [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). We'll load the dataset using [the `MySarcasmCorpus(...)` method](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MySarcasmCorpus) which returns [a `MySarcasmRecordCorpusModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySarcasmRecordCorpusModel) with the fields:
* The `records::Dict{Int, MySarcasmRecordModel}` field holds the original records data as a dictionary, where the keys of the dictionary correspond to the headline index, and the values are [instances of the `MySarcasmRecordModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySarcasmRecordModel). Each record has the following fields:
    * `issarcastic`: has a value of `1` if the record is sarcastic; otherwise, `0.`
    * `headline`: the headline of the article, unstructured text
    * `article_link`: link to the original news article. Useful in collecting supplementary data

* The `tokens::Dict{String, Int64}` field holds the vocabulary computed over the __entire dataset__ as a dictionary, where the dictionary's keys are the tokens (words) and the values of the index of the word. We assemble the `tokens` dictionary in alphabetical order. 
* The `inverse::Dict{Int64, String}` field is the inverse of the `tokens` dictionary, where the keys are the token indexes and the values are the tokens (words).

Let's call the `MySarcasmCorpus(...)` method to load the sarcasm dataset and assign it to the `corpusmodel:MySarcasmRecordCorpusModel` variable. 

In [42]:
corpusmodel = MySarcasmCorpus(); # this loads the corpus model, which contains the vocabulary and tokenization information

Each [`MySarcasmRecordModel` instance](src/Types.jl) has the three fields in the original data records: an `issarcastic::Bool` field holding the label for this record, the `headline::String` field holding the headline and the `article::String` field holding a link to the original article.

In [43]:
corpusmodel.records[5].headline

"mother comes pretty close to using word streaming correctly"

Let's explore how the tokenization system works by examining individual tokens and their mappings. The `tokens` dictionary maps words to unique integer indices, while the `inverse` dictionary allows us to look up words from their indices. 

For example, let's look at the special `<bos>` (beginning of sequence) token, which is automatically added to the start of every tokenized headline:

In [44]:
corpusmodel.tokens["<bos>"]

29662

The last _non-control_ token in the vocabulary is one before the `<bos>` token:

In [45]:
corpusmodel.inverse[29661] # this is the last "non-control" token in the vocabulary

"ünited"

___

## Tokenize the headline records
In this task, we'll use the corpus model, particularly the `tokens::Dict{String, Int64}` dictionary, to tokenize headlines in our dataset, i.e., convert a text representation into a numerical vector representation. 

> __Idea__: To better understand how this works, let's first examine a single (random) record and tokenize it.  We'll select a random record from the `number_of_records::Int64` possible records [using the built-in `rand(...)` method](https://docs.julialang.org/en/v1/stdlib/Random/#Base.rand), and store it in the `random_test_record::MySarcasmRecordModel` variable

Select a random record:

In [46]:
random_test_record = let 
    number_of_records = length(corpusmodel.records); # how many records do we have?
    random_index = rand(1:number_of_records);          # pick a random index
    corpusmodel.records[random_index]                  # get the record at that index
end

MySarcasmRecordModel(false, "reverse crowdfundgineering five ways to integrate events into your crowdfunding campaign", "https://www.huffingtonpost.com/entry/reverse-crowdfundingginee_b_5049797.html")

Call [the `tokenize(...)` method exported by the `VLDataScienceMachineLearningPackage.jl` package](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/text/#VLDataScienceMachineLearningPackage.tokenize), which takes the `headline::String` that we want to tokenize, and our vocabulary stored in the `tokens::Dict{String, Int64}` dictionary and returns a token vector.

What happens when we tokenize the headline of the `random_test_record::MySarcasmRecordModel` record?

In [47]:
tv = tokenize(random_test_record.headline, corpusmodel.tokens)

13-element Vector{Int64}:
 29662
 22355
  6767
 10339
 28768
 26824
 13860
  9429
 13966
 29543
  6768
  4485
 29663

Ok, so we get back a vector of integers, where each integer corresponds to a token in the `tokens::Dict{String, Int64}` dictionary. Let's show this in a table format, where the first column is the token index and the second column is the token itself.

`Unhide` the code cell below to see how we generated a token table using [the `pretty_table(...)` method exported by the `PrettyTables.jl` package](https://github.com/ronisbr/PrettyTables.jl).

In [48]:
let

    # initialize -
    df = DataFrame(); # holds the text data

    for i ∈ eachindex(tv)
        
        # get the token
        token = corpusmodel.inverse[tv[i]]; # use the inverse dictionary to get the token string from the token index

        # add to the dataframe
        row_data = (
            token_index = i,
            token = tv[i],
            word = token
        )
        push!(df, row_data);
    end

    # show the dataframe using PrettyTables -
    pretty_table(df, header = ["Token Index", "Token", "Word"], tf = tf_simple)
end

============== ======= =====================
  Token Index   Token                 Word 
============== ======= =====================
            1   29662                <bos>
            2   22355              reverse
            3    6767   crowdfundgineering
            4   10339                 five
            5   28768                 ways
            6   26824                   to
            7   13860            integrate
            8    9429               events
            9   13966                 into
           10   29543                 your
           11    6768         crowdfunding
           12    4485             campaign
           13   29663                <eos>
============== ======= =====================


### Hmmm. What happens if a token is not in the dataset?
We have created the vocabulary in the `tokens::Dict{String, Int64}` dictionary by analyzing the entire dataset, but suppose we have new samples that aren't in the dataset; what happens then? We've added the `<unk>` token to our dataset,  let's see how this works.

> __Idea:__ Let's take the headline from the `random_test_record::MySarcasmRecordModel` instance and add random text to the end, e.g., `#ilovemyroomba`. When we tokenize the updated headline, we should get the `<unk>` token at the end of the token vector (before the `<eos>` token).

First, to make sure we are doing what we think we are doing, let's check that `#ilovemyroomba` is __not__ already in the `tokens::Dict{String, Int64}` dictionary; if it is, [we'll get an `AssertionError`](https://docs.julialang.org/en/v1/base/base/#Core.AssertionError) when we run the code cell below.

In [49]:
words = corpusmodel.tokens |> keys |> collect; # what?? We are getting keys (words) and turning into an array
@assert ("#ilovemyroomba" ∈ words) == false # fancy way of checking if item is in vocabulary

Create a new headline by appending `#ilovemyroomba` to the old headline. String append operations in Julia use [the `*` method. ](https://docs.julialang.org/en/v1/manual/strings/) We'll store the new headline in the `new_test_headline::String` variable.

In [50]:
new_test_headline = random_test_record.headline * " " * "#ilovemyroomba"

"reverse crowdfundgineering five ways to integrate events into your crowdfunding campaign #ilovemyroomba"

Now, let's tokenize the `new_test_headline::String`, and see what happens. We save the tokenized vector for the updated headline in the `tv_new::Vector{Int64}` variable:

In [51]:
tv_new = tokenize(new_test_headline, corpusmodel.tokens)

14-element Vector{Int64}:
 29662
 22355
  6767
 10339
 28768
 26824
 13860
  9429
 13966
 29543
  6768
  4485
 29666
 29663

`Unhide` the code cell below to see how we generated a token table for the `tv_new::Vector{Int64}` using [the `pretty_table(...)` method exported by the `PrettyTables.jl` package](https://github.com/ronisbr/PrettyTables.jl).

In [ ]:
let

    # initialize -
    df = DataFrame(); # holds the text data

    for i ∈ eachindex(tv_new)
        
        # get the token
        token = corpusmodel.inverse[tv_new[i]]; # use the inverse dictionary to get the token string from the token index

        # add to the dataframe
        row_data = (
            token_index = i,
            token = tv_new[i],
            word = token
        )
        push!(df, row_data);
    end

    # show the dataframe using PrettyTables -
    pretty_table(df, header = ["Index", "Token ID", "Word"], tf = tf_simple)
end

============== ======= =====================
  Token Index   Token                 Word 
============== ======= =====================
            1   29662                <bos>
            2   22355              reverse
            3    6767   crowdfundgineering
            4   10339                 five
            5   28768                 ways
            6   26824                   to
            7   13860            integrate
            8    9429               events
            9   13966                 into
           10   29543                 your
           11    6768         crowdfunding
           12    4485             campaign
           13   29666                <unk>
           14   29663                <eos>
============== ======= =====================


Notice that the penultimate token in the `tv_new::Vector{Int64}` vector is the `<unk>` token, which corresponds to the `#ilovemyroomba` text that we appended to the original headline. The last token is the `<eos>` token, which indicates the end of the sequence.

> __Where would this be useful?__ In real-world applications, out-of-vocabulary (OOV) words are extremely common. Consider these scenarios: new product names (like "iPhone" when it was first released), trending hashtags, emerging slang, proper nouns, technical jargon, or even simple typos. A robust tokenization system must handle these gracefully without crashing. 
> 
> The `<unk>` token serves as a safety net, allowing our model to continue processing while acknowledging that it encountered something unfamiliar. This is particularly important when deploying models in production, where the input data will inevitably contain words that weren't present during training.

Cool! So, how do we handle varying lengths of token sequences?

### Compute the maximum pad length
Not every headline has the same length, but we want the token vectors to have the same size. Thus, we'll find the longest vectors in the dataset and pad the token vectors to that length. However, before we address sequence length, let's briefly consider the vocabulary we've created. Our corpus contains thousands of unique words, each mapped to a distinct integer. This raises an important design question: __how large should our vocabulary be?__

> __Understanding Vocabulary Size and Trade-offs.__
> A larger vocabulary captures more linguistic nuance, e.g., it can distinguish between "happy," "joyful," and "ecstatic" as separate concepts. However, larger vocabularies also mean:
> * More memory requirements for storing token mappings
> * Larger set of model parameters (e.g., each word needs its own learned representation when we construct embeddings)
> * Potential overfitting to rare words that appear infrequently
>
> Conversely, a smaller vocabulary is more computationally efficient but might lose important distinctions by mapping different words to the same `<unk>` token.
>
> The optimal vocabulary size depends on your specific application, dataset characteristics, and computational constraints. For this sarcasm dataset, we can check our actual vocabulary size and see how it compares to the number of unique headlines.

In [53]:
println("Our sarcasm dataset vocabulary contains $(length(corpusmodel.tokens)) unique tokens")
println("The dataset contains $(length(corpusmodel.records)) headlines")

Our sarcasm dataset vocabulary contains 29667 unique tokens
The dataset contains 28619 headlines


> __Why a dictionary?__ Finally, the choice of using dictionaries for our token mappings (rather than arrays) reflects the need for fast lookup operations, we frequently need to find a word's index or convert an index back to a word, operations that dictionaries handle efficiently.

Ok. Back to finding the pad length. First, iterate through each headline, compute its size, and then save this length if it is longer than we've seen before. We'll print out some information along the way to see how this works.

In [54]:
max_pad_length = let

    # initialize -
    number_of_records = corpusmodel.records |> length; # how many records do we have?
    max_pad_length = 0; # initialize: we have 0 length
    
    # test the length of each headline
    for i ∈ 1:number_of_records
        test_record_length = tokenize(corpusmodel.records[i].headline, corpusmodel.tokens) |> length; # tokenize, and calc the number of tokens
        if (test_record_length > max_pad_length)
            max_pad_length = test_record_length; # we've found a new longest headline!
            println("Found a new longest headline: $(i) with length: $(max_pad_length)"); # show the record number and length
        end
    end
    max_pad_length
end;

Found a new longest headline: 1 with length: 10
Found a new longest headline: 2 with length: 15
Found a new longest headline: 11 with length: 16
Found a new longest headline: 14 with length: 18
Found a new longest headline: 37 with length: 20
Found a new longest headline: 97 with length: 21
Found a new longest headline: 106 with length: 22
Found a new longest headline: 189 with length: 23
Found a new longest headline: 584 with length: 24
Found a new longest headline: 1238 with length: 25
Found a new longest headline: 1450 with length: 26
Found a new longest headline: 2147 with length: 31
Found a new longest headline: 7303 with length: 153


So, the maximum length headline is:

In [55]:
println("The maximum pad length is: $(max_pad_length) tokens.")

The maximum pad length is: 153 tokens.


Now that we know the maximum length of the token vectors, we can pad the shorter vectors with the `<pad>` token to ensure that all vectors __have the same length__. This is important for training machine learning models, as they typically require fixed-size input vectors.

> __Note on alternatives:__ While padding to a fixed length is the most straightforward approach, other methods exist for handling variable-length sequences. These include dynamic batching (grouping sequences of similar lengths), attention masking (teaching models to ignore padding tokens), and recurrent architectures that can naturally handle sequences of varying lengths. For this introduction, fixed-length padding provides the clearest path to understanding the fundamental concepts.

We'll use `right-padding` (adding padding tokens at the end) and will store the tokenized records for each headline in the `token_record_dictionary::Dict{Int64, Array{Int64,1}}` dictionary, where the keys of this dictionary are the record indexes, and the values are the tokenized records (which are of type `Array{Int64,1}`).

In [56]:
token_record_dictionary = let

    # initialize -
    number_of_records = corpusmodel.records |> length; # how many records do we have?
    token_record_dictionary = Dict{Int64, Array{Int64,1}}(); # dictionary to hold the tokenized records

    # tokenize each record, store result in the dictionary
    for i ∈ 1:number_of_records
        token_record_dictionary[i] = tokenize(corpusmodel.records[i].headline, corpusmodel.tokens, 
                pad = max_pad_length); # tokenize, and pad to max length
    end

    token_record_dictionary;
end

Dict{Int64, Vector{Int64}} with 28619 entries:
  24824 => [29662, 25875, 6521, 16122, 24450, 13456, 7182, 19560, 4735, 29665  …
  25754 => [29662, 20178, 17480, 12830, 18533, 19764, 25505, 3015, 20257, 28436…
  11950 => [29662, 2444, 8038, 4360, 1643, 6928, 18871, 21115, 29665, 29665  … …
  1703  => [29662, 8234, 6705, 23705, 26824, 29321, 16578, 18613, 4066, 23170  …
  12427 => [29662, 17645, 22221, 26824, 12325, 20943, 29190, 28512, 21850, 8481…
  7685  => [29662, 26616, 26361, 27360, 26824, 16115, 26532, 22566, 22965, 2948…
  18374 => [29662, 10011, 18294, 16584, 29536, 18113, 12255, 23170, 8927, 28691…
  3406  => [29662, 11739, 15062, 1641, 6106, 26532, 20433, 18531, 6328, 13456  …
  23970 => [29662, 5537, 6659, 17966, 28343, 26824, 13628, 19950, 26729, 18531 …
  27640 => [29662, 16774, 18531, 1231, 11339, 3999, 26855, 15141, 5731, 18531  …
  28576 => [29662, 2834, 21826, 26824, 1062, 19441, 21861, 25273, 29665, 29665 …
  1090  => [29662, 5537, 1811, 27110, 15250, 24021, 23868, 291

Let's take a look at two example records, the first record in the dataset and the last record in the dataset, and see how they look after tokenization and padding (we'll put these in array format for easy viewing):

In [57]:
example_array = let 
    number_of_records = corpusmodel.records |> length; # how many records do we have?
    first_record = token_record_dictionary[1];
    last_record = token_record_dictionary[number_of_records];
    [first_record last_record]
end

154×2 Matrix{Int64}:
 29662  29662
 26615   6988
 23293   5430
 27978  26616
  8293  18269
  5551    912
 18531  10569
 12045  25421
 15826  29665
 29665  29665
     ⋮  
 29665  29665
 29665  29665
 29665  29665
 29665  29665
 29665  29665
 29665  29665
 29665  29665
 29665  29665
 29663  29663

In [58]:
corpusmodel.tokens["<pad>"]

29665

__What do we see?__ The records are on the columns, while the tokens are on the rows. 

* Both records have the same length, which is the maximum length of the token vectors. The shorter record has been padded with the `<pad> = 29665` token to match the length of the longer record. 

* The first, and last tokens in each record are the `<bos>` and `<eos>` tokens, respectively, which indicate the start and end of the sequence. The other tokens are the actual words from the headlines, with the `<unk>` token appearing where a word was not found in the vocabulary. 

Interesting! Let's dump this to disk so we can use it later. 

## Final: Save data to disk
We did a bunch of stuff in this example, and we don't want to have to recompute the corpus, token dictionary, etc. So let's save it [in an HDF5 encoded binary file](https://en.wikipedia.org/wiki/Hierarchical_Data_Format). 

To start, specify a path in the `path_to_save_file::String` variable where we want to save the data:

In [59]:
path_to_save_file = joinpath(_PATH_TO_DATA, "CHEME-141-M4-SarcasmSamplesTokenizer-SavedData.jld2"); # JLD2 package encodes data

Next, write data to disk as a `jld2` (binary) saved file using [the `save(...)` method exported by the FileIO.jl package](https://github.com/JuliaIO/FileIO.jl).  This will save the data as a [Julia `Dict` type](https://docs.julialang.org/en/v1/base/collections/#Base.Dict). The save file is [an HDF5 encoded file format](https://en.wikipedia.org/wiki/Hierarchical_Data_Format), which is small (compressed), which is excellent! 

In [60]:
save(path_to_save_file, Dict("corpus" => corpusmodel)); # encode, and write

___